In [ ]:
#| default_exp plotting

In [ ]:
#| export
"""Small, dataframe-first visualizations for ProbeStore objects."""
from __future__ import annotations

import math
import json
from pathlib import Path
import re
import pandas as pd


def _frame(data) -> pd.DataFrame:
    frame = getattr(data, "obs", data)
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("data must be a ProbeStore or pandas DataFrame")
    return frame


def _plt():
    try:
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise RuntimeError("Plotting requires matplotlib; install probeharbour[plot]") from exc
    return plt


def _empty_figure(message: str, figsize):
    plt = _plt()
    fig, ax = plt.subplots(figsize=figsize)
    ax.text(.5, .5, message, ha="center", va="center", transform=ax.transAxes)
    ax.axis("off")
    return fig


def _save_figure(fig, save, *, dpi: int, transparent: bool) -> None:
    if save is None:
        return
    destination = Path(save)
    destination.parent.mkdir(parents=True, exist_ok=True)
    output_format = destination.suffix.lower().lstrip(".") or None
    fig.savefig(destination, format=output_format, dpi=dpi,
                transparent=transparent, bbox_inches="tight")


def _present(value, *, digits: int | None = None, suffix: str = "") -> str:
    """Format optional record-card evidence without inventing missing results."""
    if value is None or (not isinstance(value, (list, dict, tuple)) and pd.isna(value)):
        return "not calculated"
    if digits is not None:
        try:
            return f"{float(value):.{digits}f}{suffix}"
        except (TypeError, ValueError):
            pass
    return f"{value}{suffix}"


def _value(row: pd.Series, *names, default=None):
    for name in names:
        if name in row:
            value = row[name]
            if isinstance(value, (list, dict, tuple)) or (value is not None and bool(pd.notna(value))):
                return value
    return default


def _status(value, *, positive_is_pass=True) -> str:
    if value is None or pd.isna(value):
        return "not evaluated"
    passed = bool(value) if positive_is_pass else not bool(value)
    return "pass" if passed else "flagged"


def _modifier_status(row, modifier, *, pass_field=None, fail_field=None):
    if pass_field and _value(row, pass_field) is not None:
        return _status(_value(row, pass_field))
    if fail_field and _value(row, fail_field) is not None:
        return _status(_value(row, fail_field), positive_is_pass=False)
    modifiers = _value(row, "probeharbour_score_modifiers", "recalculated_score_modifiers",
                       "score_modifiers")
    if modifiers is None:
        return "not evaluated"
    if isinstance(modifiers, str):
        modifiers = [modifiers]
    return "flagged" if modifier in set(modifiers) else "pass"


def _ligation_join(rbd5: str, rbd3: str, stored=None):
    """Return the observable arm-end ligation bases and any consistency warning."""
    if not rbd5 or not rbd3:
        return None, None
    derived = f"{rbd5[-1]}{rbd3[0]}".upper()
    warning = None
    if stored is not None and not pd.isna(stored):
        normalized = str(stored).replace("|", "").replace("/", "").replace(" ", "").upper()
        if normalized != derived:
            warning = f"stored {stored}; sequence-derived {derived}"
    return f"{derived[0]} | {derived[1]}", warning


def _draw_binding_arms(ax, *, x, y, width, height, rbd5, rbd3,
                       show_labels=True, sequence_font=8):
    """Draw two arms and mark the true ligation boundary without duplicating bases."""
    from matplotlib.patches import FancyBboxPatch
    gap = .012
    arm_width = (width - gap) / 2
    for offset, sequence, colour, label in (
        (0, rbd5, "#2d5fa6", "5′ arm (RBD5)"),
        (arm_width + gap, rbd3, "#c05820", "3′ arm (RBD3)"),
    ):
        left = x + offset
        ax.add_patch(FancyBboxPatch(
            (left, y), arm_width, height,
            boxstyle="round,pad=.004,rounding_size=.007",
            facecolor=colour, edgecolor="white", lw=1,
        ))
        ax.text(left + arm_width / 2, y + height / 2, sequence or "not available",
                ha="center", va="center", fontsize=sequence_font, color="white",
                fontfamily="monospace", fontweight="bold")
        if show_labels:
            ax.text(left + arm_width / 2, y - .012, label, ha="center", va="top",
                    fontsize=7, color="#61727b")
    boundary = x + arm_width + gap / 2
    ax.plot([boundary, boundary], [y - .004, y + height + .004], color="#0f766e", lw=2.2,
            solid_capstyle="round", zorder=5)
    return boundary


def _short_identifier(value, limit=30):
    text = _present(value)
    if len(text) <= limit:
        return text
    keep = max(6, (limit - 1) // 2)
    return f"{text[:keep]}…{text[-keep:]}"


def _readable_items(value, labels):
    if value is None:
        return "none"
    if isinstance(value, str):
        value = [value]
    elif hasattr(value, "tolist"):
        value = value.tolist()
    elif not isinstance(value, (list, tuple, set)):
        if pd.isna(value):
            return "none"
        value = [value]
    return "; ".join(labels.get(str(item), str(item).replace("_", " ")) for item in value) or "none"


def _probe_label(row: pd.Series, *, label_mode: str) -> str:
    name = row.get("probe_name")
    if pd.isna(name):
        name = row.get("candidate_id", "probe")
    name = str(name)
    role = row.get("panel_role")
    role = None if pd.isna(role) else str(role)
    if label_mode == "full":
        return f"{name} [{role}]" if role else name
    if role == "existing":
        codeword = row.get("codeword")
        if pd.notna(codeword):
            return f"10x · codeword {int(codeword)}"
        return re.sub(r"^panel_", "10x · ", name) if name.startswith("panel_") else f"10x · {name}"
    match = re.search(r":J(\d+):slide:([+-]\d+)$", name)
    if match:
        role_name = {"add_on": "add-on", "designed": "designed"}.get(role, role or "designed")
        return f"{role_name} · J{match.group(1)} · {match.group(2)} nt"
    role_name = {"add_on": "add-on", "custom_control": "control"}.get(role, role)
    return f"{name} · {role_name}" if role_name else name


def _parse_junction_id(value):
    match = re.match(r"^(?:chr)?([^:]+):(\d+)-(\d+):([+-])$", str(value))
    if not match:
        return None
    return match.group(1), int(match.group(2)), int(match.group(3)), match.group(4)


def _transcript_label(label: str, strand: str) -> str:
    orientation = "5′ → 3′ · positive strand" if strand == "+" else "3′ ← 5′ · negative strand"
    return f"{label} · {orientation}"


def _merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


def _compressed_locus_transform(exons):
    """Return an exon-preserving transform with balanced logarithmic intron gaps."""
    merged = _merge_intervals(exons)
    locus_left, locus_right = merged[0][0], merged[-1][1]
    exon_bases = sum(end - start for start, end in merged)
    intron_bases = max((locus_right - locus_left) - exon_bases, 0)
    exon_lengths = sorted(end - start for start, end in merged)
    median_exon = exon_lengths[len(exon_lengths) // 2]
    compression_ratio = min(1.0, exon_bases / intron_bases) if intron_bases else 1.0
    pieces = []
    cursor, display = locus_left, 0.0
    for start, end in merged:
        if start > cursor:
            intron_length = start - cursor
            # Logarithmic gaps preserve ordering and relative scale while
            # preventing a megabase intron from hiding neighbouring exons.
            desired_gap = median_exon * (1.5 + math.log10(max(intron_length / median_exon, 1.0)))
            intron_scale = min(1.0, desired_gap / intron_length)
            pieces.append((cursor, start, intron_scale, display))
            display += intron_length * intron_scale
        pieces.append((start, end, 1.0, display))
        display += end - start
        cursor = end

    def transform(value):
        value = min(max(float(value), locus_left), locus_right)
        for start, end, scale, offset in pieces:
            if value <= end:
                return offset + (value - start) * scale
        return display

    return transform, locus_left, locus_right, compression_ratio

In [ ]:
#| export
def panel_overview(data, *, title: str | None = None, figsize=(10, 7)):
    """Plot gene coverage, junction geometry, GC, and ligation composition."""
    plt = _plt()
    frame = _frame(data).copy()
    if frame.empty:
        return _empty_figure("No probes", figsize)
    gene_column = next((column for column in ("gene_name", "gene", "gene_id") if column in frame), None)
    if "binder" not in frame and {"rbd5", "rbd3"} <= set(frame):
        frame["binder"] = frame["rbd5"].astype(str) + frame["rbd3"].astype(str)
    if "gc_percent" not in frame and "binder" in frame:
        frame["gc_percent"] = frame["binder"].map(
            lambda sequence: 100 * (sequence.upper().count("G") + sequence.upper().count("C")) / len(sequence)
            if sequence else math.nan
        )
    if "lig_dinuc" not in frame and {"rbd5", "rbd3"} <= set(frame):
        frame["lig_dinuc"] = frame["rbd5"].str[-1] + frame["rbd3"].str[0]

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    if gene_column:
        frame.groupby(gene_column).size().plot.hist(ax=axes[0, 0], bins=30, color="#2563eb")
        axes[0, 0].set(xlabel="probes per gene", ylabel="genes", title="Gene coverage")
    else:
        axes[0, 0].text(.5, .5, "No gene column", ha="center", va="center", transform=axes[0, 0].transAxes)
    if "junction_spanning" in frame:
        counts = frame["junction_spanning"].fillna(False).map({False: "continuous", True: "junction-spanning"}).value_counts()
        axes[0, 1].bar(counts.index, counts.values, color=["#94a3b8", "#7c3aed"][:len(counts)])
        axes[0, 1].tick_params(axis="x", rotation=15)
        axes[0, 1].set(ylabel="probes", title="BED geometry")
    else:
        axes[0, 1].text(.5, .5, "No junction geometry", ha="center", va="center", transform=axes[0, 1].transAxes)
    if "gc_percent" in frame:
        frame["gc_percent"].dropna().plot.hist(ax=axes[1, 0], bins=30, color="#16a34a")
        axes[1, 0].set(xlabel="GC (%)", ylabel="probes", title="Target-region GC")
    if "lig_dinuc" in frame:
        frame["lig_dinuc"].value_counts().sort_index().plot.bar(ax=axes[1, 1], color="#ea580c")
        axes[1, 1].set(xlabel="dinucleotide", ylabel="probes", title="Ligation bases")
    for ax in axes.flat:
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(title or getattr(data, "name", "Panel overview"), fontweight="bold")
    fig.tight_layout()
    return fig

In [ ]:
#| export
def thermodynamics(data, *, title="Probe thermodynamics", figsize=(10, 7)):
    """Plot arm balance, full Tm, folding energies, and repeat prevalence."""
    plt = _plt()
    frame = _frame(data)
    required = {"tm_5p", "tm_3p", "tm_full"}
    if frame.empty or not required <= set(frame):
        return _empty_figure("Run store.characterize() before plotting thermodynamics", figsize)
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes[0, 0].scatter(frame["tm_5p"], frame["tm_3p"], s=10, alpha=.35, color="#2563eb")
    lo = min(frame["tm_5p"].min(), frame["tm_3p"].min())
    hi = max(frame["tm_5p"].max(), frame["tm_3p"].max())
    axes[0, 0].plot([lo, hi], [lo, hi], "--", color="#94a3b8", linewidth=1)
    axes[0, 0].set(xlabel="RBD5 Tm (°C)", ylabel="RBD3 Tm (°C)", title="Arm balance")
    frame["tm_full"].dropna().plot.hist(ax=axes[0, 1], bins=35, color="#7c3aed")
    axes[0, 1].set(xlabel="Tm (°C)", ylabel="probes", title="Full target-region Tm")
    energy_columns = [column for column in ("p3_hairpin_dg", "p3_homodimer_dg") if column in frame]
    for column, color in zip(energy_columns, ("#0891b2", "#d97706")):
        frame[column].dropna().plot.hist(ax=axes[1, 0], bins=35, alpha=.55, label=column, color=color)
    axes[1, 0].legend(frameon=False)
    axes[1, 0].set(xlabel="ΔG (kcal/mol)", ylabel="probes", title="Secondary structure")
    repeat_columns = [column for column in ("has_simple_repeat", "has_5g_repeat") if column in frame]
    values = [int(frame[column].fillna(False).astype(bool).sum()) for column in repeat_columns]
    axes[1, 1].bar([column.replace("has_", "") for column in repeat_columns], values, color="#be185d")
    axes[1, 1].set(ylabel="flagged probes", title="Sequence warnings")
    for ax in axes.flat:
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(title, fontweight="bold")
    fig.tight_layout()
    return fig

In [ ]:
#| export
def junction_map(gene: str, transcript_data, junctions, *, selected_junctions=(),
                 transcript_ids=None, renumber_junctions: bool = False,
                 max_transcripts: int = 20, overview_scale: str = "auto",
                 title: str | None = None, figsize=None,
                 save: str | Path | None = None, dpi: int = 300,
                 transparent: bool = False):
    """Plot transcript models and selectable junctions coloured by prevalence."""
    plt = _plt()
    from matplotlib.path import Path as MplPath
    from matplotlib.patches import PathPatch
    frame = _frame(junctions).copy()
    if "junction_id" not in frame:
        raise ValueError("junctions requires a junction_id column")
    frame = frame.drop_duplicates("junction_id")
    frame["_parsed"] = frame["junction_id"].map(_parse_junction_id)
    frame = frame.loc[frame["_parsed"].notna()].copy()
    if frame.empty:
        return _empty_figure(f"No mapped junctions for {gene}", figsize or (10, 3))
    if renumber_junctions or "junction_label" not in frame:
        frame = frame.sort_values("_parsed").reset_index(drop=True)
        frame["junction_label"] = [f"J{index + 1}" for index in range(len(frame))]
    class_column = "junction_class" if "junction_class" in frame else "junction_specificity_class"
    if class_column not in frame:
        frame[class_column] = "unclassified"
    selected = {str(value) for value in selected_junctions}
    available_transcripts = list(transcript_data.get_transcripts_by_gene_name(gene))
    if not available_transcripts:
        available_transcripts = list(transcript_data.get_transcripts_by_gene_id(gene))
    if transcript_ids is not None:
        requested = {str(value) for value in transcript_ids}
        available_transcripts = [value for value in available_transcripts if str(value) in requested]
    models = []
    for transcript_id in available_transcripts:
        exon_coords, strand_code = transcript_data.get_exon_coords_and_strand(transcript_id)
        exons = sorted((min(pair), max(pair)) for pair in exon_coords)
        if not exons: continue
        info = transcript_data.get_transcript_info(transcript_id)
        models.append((str(info.get("transcript_name") or transcript_id), exons,
                       "+" if strand_code in {1, "+"} else "-"))
    models = models[:max_transcripts]
    if not models:
        return _empty_figure(f"No transcript models for {gene}", figsize or (10, 3))
    all_exons = [exon for _, exons, _ in models for exon in exons]
    transform, locus_left, locus_right, intron_scale = _compressed_locus_transform(all_exons)
    compressed = overview_scale == "compressed" or (overview_scale == "auto" and intron_scale < .08)
    display = transform if compressed else lambda value: float(value)
    shown_left, shown_right = display(locus_left), display(locus_right)
    span = max(shown_right - shown_left, 1)
    fig, ax = plt.subplots(figsize=figsize or (12, max(3.2, .52 * len(models) + 1.7)), layout="constrained")
    colours = {"specific": "#0f766e", "partially specific": "#d97706",
               "partially_specific": "#d97706", "constitutive": "#64748b",
               "unclassified": "#94a3b8"}
    labelled = set()
    for y, (label, exons, strand) in enumerate(models[::-1]):
        ax.plot([display(exons[0][0]), display(exons[-1][1])], [y, y], color="#cbd5e1", lw=1.2)
        for start, end in exons:
            ax.add_patch(plt.Rectangle((display(start), y - .12), display(end) - display(start), .24,
                                       facecolor="#475569", edgecolor="none", zorder=2))
        ax.text(shown_left + .006 * span, y + .2, _transcript_label(label, strand),
                fontsize=7.2, ha="left", va="bottom", color="#1f2937")
        for _, junction in frame.iterrows():
            _, jstart, jend, jstrand = junction["_parsed"]
            if strand != jstrand: continue
            pair = next(((first, second) for first, second in zip(exons, exons[1:])
                         if abs(first[1] - jstart) <= 1 and abs(second[0] - jend) <= 1), None)
            if pair is None: continue
            left, right = display(pair[0][1]), display(pair[1][0])
            chosen = str(junction["junction_id"]) in selected
            class_colour = colours.get(str(junction[class_column]).lower(), "#94a3b8")
            apex = y + (.34 if y % 2 == 0 else -.34)
            path = MplPath([(left, y), ((left + right) / 2, apex), (right, y)],
                           [MplPath.MOVETO, MplPath.CURVE3, MplPath.CURVE3])
            ax.add_patch(PathPatch(path, facecolor="none", edgecolor=class_colour,
                                   lw=1.8 if chosen else 1.4,
                                   alpha=1 if chosen else .8, zorder=4))
            if chosen:
                ax.scatter((left + right) / 2, apex, color="#c026d3", s=22,
                           edgecolor="white", linewidth=.5, zorder=6)
            junction_id = str(junction["junction_id"])
            if junction_id not in labelled:
                ax.text((left + right) / 2, apex, str(junction["junction_label"]), ha="center",
                        va="bottom" if apex >= y else "top", fontsize=7.2, fontweight="bold",
                        color=class_colour,
                        bbox={"facecolor": "white", "edgecolor": "none", "alpha": .85, "pad": 1})
                labelled.add(junction_id)
    ax.set(xlim=(shown_left, shown_right), ylim=(-.7, len(models) - .25), yticks=[])
    ax.spines[["top", "right", "left"]].set_visible(False)
    if compressed:
        ax.set_xticks([]); ax.set_xlabel("Schematic locus (introns compressed; exon widths preserved)")
    else:
        from matplotlib.ticker import FuncFormatter
        ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{int(value):,}"))
        ax.set_xlabel("Genomic coordinate (0-based; bp)")
    ax.set_title(title or f"{gene}: annotated transcripts and selectable junctions",
                 loc="left", fontweight="bold", pad=18)
    handles = [plt.Line2D([0], [0], color="#475569", lw=5, label="exon")]
    handles += [plt.Line2D([0], [0], color=colour, lw=2, label=label) for label, colour in
                (("specific", "#0f766e"), ("partially specific", "#d97706"),
                 ("constitutive", "#64748b"))]
    handles.append(plt.Line2D([0], [0], color="#c026d3", marker="o", lw=0,
                              label="selected (class colour retained)"))
    ax.legend(handles=handles, frameon=False, ncol=5, loc="lower left",
              bbox_to_anchor=(0, 1.01), borderaxespad=0)
    _save_figure(fig, save, dpi=dpi, transparent=transparent)
    return fig

In [ ]:
#| export
def probe_map(data, *, gene: str | None = None, max_probes: int = 30,
              title: str | None = None, figsize=None):
    """Plot mapped RBD5/RBD3 genomic intervals for one gene or probe subset."""
    plt = _plt()
    frame = _frame(data).copy()
    gene_column = next((column for column in ("gene_name", "gene", "gene_id") if column in frame), None)
    if gene is not None:
        if gene_column is None:
            raise ValueError("gene was supplied but no gene column is present")
        frame = frame.loc[frame[gene_column].astype(str).eq(str(gene))]
    columns = {"rbd5_chrom_start", "rbd5_chrom_end", "rbd3_chrom_start", "rbd3_chrom_end"}
    if frame.empty or not columns <= set(frame):
        return _empty_figure("No mapped probe coordinates", figsize or (9, 2.5))
    frame = frame.dropna(subset=list(columns)).head(max_probes).copy()
    if frame.empty:
        return _empty_figure("No mapped probe coordinates", figsize or (9, 2.5))
    name_column = next((column for column in ("probe_name", "candidate_id") if column in frame), None)
    height = max(2.5, .35 * len(frame) + 1.2)
    fig, ax = plt.subplots(figsize=figsize or (10, height))
    for y, (_, row) in enumerate(frame.iloc[::-1].iterrows()):
        ax.plot([row["rbd5_chrom_start"], row["rbd5_chrom_end"]], [y, y], color="#2d5fa6", lw=6, solid_capstyle="butt")
        ax.plot([row["rbd3_chrom_start"], row["rbd3_chrom_end"]], [y, y], color="#c05820", lw=6, solid_capstyle="butt")
        inner = sorted((row["rbd5_chrom_start"], row["rbd5_chrom_end"], row["rbd3_chrom_start"], row["rbd3_chrom_end"]))[1:3]
        ax.plot(inner, [y, y], color="#94a3b8", lw=.8, ls=":", zorder=0)
    labels = (frame.iloc[::-1][name_column].astype(str).tolist() if name_column
              else [str(index) for index in frame.iloc[::-1].index])
    ax.set_yticks(range(len(frame)), labels)
    ax.set(xlabel="genomic coordinate (0-based)", title=title or (f"{gene} probe map" if gene else "Probe map"))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.legend(handles=[
        plt.Line2D([0], [0], color="#2d5fa6", lw=6, label="RBD5"),
        plt.Line2D([0], [0], color="#c05820", lw=6, label="RBD3"),
    ], frameon=False, loc="best")
    fig.tight_layout()
    return fig

In [ ]:
#| export
def gene_map(gene: str, transcript_data, probes=None, *, max_transcripts: int = 20,
             transcript_scope: str = "auto", focus: str = "auto", padding: float = .08,
             label_mode: str = "short", show_overview: bool = True,
             overview_scale: str = "auto",
             title: str | None = None, figsize=None, save: str | Path | None = None,
             dpi: int = 300, transparent: bool = False):
    """Plot transcript exon models with readable, labelled probe overlays.

    When mapped probes are supplied, the default ``auto`` view shows supporting
    transcripts and zooms to the probed region. Pass ``transcript_scope="all"``
    and ``focus="gene"`` to inspect the complete annotated locus. ``save`` may
    be a vector ``.svg``/``.pdf`` path or a raster image path such as ``.png``.
    """
    plt = _plt()
    if transcript_scope not in {"auto", "probed", "all"}:
        raise ValueError("transcript_scope must be 'auto', 'probed', or 'all'")
    if focus not in {"auto", "probes", "gene"}:
        raise ValueError("focus must be 'auto', 'probes', or 'gene'")
    if padding < 0:
        raise ValueError("padding must be non-negative")
    if label_mode not in {"short", "full"}:
        raise ValueError("label_mode must be 'short' or 'full'")
    if overview_scale not in {"auto", "compressed", "genomic"}:
        raise ValueError("overview_scale must be 'auto', 'compressed', or 'genomic'")
    transcript_ids = list(transcript_data.get_transcripts_by_gene_name(gene))
    if not transcript_ids:
        transcript_ids = list(transcript_data.get_transcripts_by_gene_id(gene))
    if not transcript_ids:
        return _empty_figure(f"Gene not found: {gene}", figsize or (10, 2.5))
    all_transcript_ids = list(transcript_ids)
    frame = _frame(probes).copy() if probes is not None else pd.DataFrame()
    if not frame.empty:
        gene_columns = [column for column in ("gene_name", "gene", "gene_id") if column in frame]
        if gene_columns:
            gene_match = pd.Series(False, index=frame.index)
            for column in gene_columns:
                gene_match |= frame[column].notna() & frame[column].astype(str).eq(str(gene))
            frame = frame.loc[gene_match]
        # Designed candidates keep coordinates in a portable nested record;
        # mapped CSV probes already expose the same arm fields as columns.
        if "coordinates" in frame:
            for arm in ("rbd5", "rbd3"):
                for edge in ("start", "end"):
                    column = f"{arm}_chrom_{edge}"
                    packed = frame["coordinates"].map(
                        lambda value: value.get(f"{arm}_{edge}")
                        if isinstance(value, dict) else None
                    )
                    frame[column] = frame[column].fillna(packed) if column in frame else packed
    coordinate_columns = (
        "rbd5_chrom_start", "rbd5_chrom_end", "rbd3_chrom_start", "rbd3_chrom_end"
    )
    if set(coordinate_columns) <= set(frame):
        numeric_arms = frame[list(coordinate_columns)].apply(pd.to_numeric, errors="coerce")
        finite_arms = numeric_arms.apply(
            lambda column: column.map(
                lambda value: pd.notna(value) and math.isfinite(float(value))
            )
        ).all(axis=1)
        frame.loc[:, list(coordinate_columns)] = numeric_arms
        mapped = frame.loc[finite_arms].copy()
    else:
        mapped = pd.DataFrame()
    if not frame.empty:
        bed_start = pd.to_numeric(
            frame.get("chrom_start", pd.Series(index=frame.index, dtype=float)), errors="coerce"
        )
        bed_end = pd.to_numeric(
            frame.get("chrom_end", pd.Series(index=frame.index, dtype=float)), errors="coerce"
        )
        finite_bed = bed_start.map(lambda value: pd.notna(value) and math.isfinite(float(value))) & bed_end.map(
            lambda value: pd.notna(value) and math.isfinite(float(value))
        )
        frame["chrom_start"], frame["chrom_end"] = bed_start, bed_end
        bed_only = frame.loc[finite_bed & ~frame.index.isin(mapped.index)].copy()
    else:
        bed_only = pd.DataFrame()
    use_probed_transcripts = transcript_scope == "probed" or (
        transcript_scope == "auto" and not mapped.empty and "transcript_id" in mapped
    )
    if use_probed_transcripts:
        supported = set(mapped["transcript_id"].dropna().astype(str))
        transcript_ids = [value for value in transcript_ids if str(value) in supported]
    transcript_ids = transcript_ids[:max_transcripts]
    if not transcript_ids:
        return _empty_figure(f"No supporting transcripts: {gene}", figsize or (10, 2.5))
    models = []
    for transcript_id in transcript_ids:
        exon_coords, strand_code = transcript_data.get_exon_coords_and_strand(transcript_id)
        info = transcript_data.get_transcript_info(transcript_id)
        exons = []
        for pair in exon_coords:
            try:
                start, end = float(pair[0]), float(pair[1])
            except (TypeError, ValueError, IndexError):
                continue
            if math.isfinite(start) and math.isfinite(end):
                exons.append((min(start, end), max(start, end)))
        exons.sort()
        if exons:
            models.append((transcript_id, str(info.get("transcript_name", transcript_id)), exons,
                           "+" if strand_code in {1, "+"} else "-"))
    if not models:
        return _empty_figure(f"No exon models: {gene}", figsize or (10, 2.5))
    use_probe_focus = focus == "probes" or (
        focus == "auto" and (not mapped.empty or not bed_only.empty)
    )
    if use_probe_focus:
        starts = []
        ends = []
        if not mapped.empty:
            starts.extend(mapped[list(coordinate_columns)].min().astype(float).tolist())
            ends.extend(mapped[list(coordinate_columns)].max().astype(float).tolist())
        if not bed_only.empty:
            starts.extend(bed_only["chrom_start"].astype(float).tolist())
            ends.extend(bed_only["chrom_end"].astype(float).tolist())
        starts = [value for value in starts if math.isfinite(value)]
        ends = [value for value in ends if math.isfinite(value)]
        if not starts or not ends:
            use_probe_focus = False
    if use_probe_focus:
        probe_left, probe_right = min(starts), max(ends)
        span = max(float(probe_right - probe_left), 1.0)
        margin = max(500.0, span * padding)
        view_left, view_right = max(0.0, probe_left - margin), probe_right + margin
    else:
        view_left = min(exon[0] for _, _, exons, _ in models for exon in exons)
        view_right = max(exon[1] for _, _, exons, _ in models for exon in exons)
    view_span = max(view_right - view_left, 1)
    main_height = max(3.0, 1.05 * len(models) + 1.4 + .3 * min(len(bed_only), 6))
    overview_enabled = bool(show_overview and use_probe_focus)
    if overview_enabled:
        overview_height = max(1.15, .58 * len(models) + .45)
        fig = plt.figure(figsize=figsize or (11, main_height + overview_height + .65), layout="constrained")
        grid = fig.add_gridspec(3, 1, height_ratios=[.46, overview_height, main_height], hspace=.04)
        overview_ax = fig.add_subplot(grid[1])
        legend_ax = fig.add_subplot(grid[0])
        ax = fig.add_subplot(grid[2])
    else:
        fig = plt.figure(figsize=figsize or (11, main_height + .45), layout="constrained")
        grid = fig.add_gridspec(2, 1, height_ratios=[.46, main_height], hspace=.02)
        legend_ax = fig.add_subplot(grid[0])
        ax = fig.add_subplot(grid[1])
        overview_ax = None
    legend_ax.axis("off")
    for y, (transcript_id, label, exons, strand) in enumerate(models[::-1]):
        line_start, line_end = max(exons[0][0], view_left), min(exons[-1][1], view_right)
        if line_start <= line_end:
            ax.plot([line_start, line_end], [y, y], color="#94a3b8", lw=1, zorder=0)
        for start, end in exons:
            ax.add_patch(plt.Rectangle((start, y - .12), end - start, .24,
                                       color="#475569", ec="none", zorder=1))
        ax.text(view_left + .008 * view_span, y + .25, _transcript_label(label, strand),
                fontsize=8, ha="left", va="bottom", clip_on=True)
        if frame.empty or "transcript_id" not in frame:
            continue
        transcript_probes = mapped.loc[mapped["transcript_id"].astype(str).eq(str(transcript_id))]
        for offset, (_, row) in enumerate(transcript_probes.head(8).iterrows()):
            py = y - .27 - .15 * offset
            # Arm bounds can cross an intron. Draw their intersections with
            # exons rather than a misleading solid line through genomic space.
            arm_segments = {}
            for arm, color in (("rbd5", "#2d5fa6"), ("rbd3", "#c05820")):
                arm_start = float(row[f"{arm}_chrom_start"])
                arm_end = float(row[f"{arm}_chrom_end"])
                segments = []
                for exon_start, exon_end in exons:
                    segment_start = max(arm_start, exon_start)
                    segment_end = min(arm_end, exon_end)
                    if (math.isfinite(segment_start) and math.isfinite(segment_end)
                            and segment_start < segment_end):
                        segments.append((segment_start, segment_end))
                arm_segments[arm] = segments
            segment_centres = [
                start + (end - start) / 2
                for segments in arm_segments.values() for start, end in segments
            ]
            segment_centres = [value for value in segment_centres if math.isfinite(value)]
            if len(segment_centres) > 1:
                ax.plot([min(segment_centres), max(segment_centres)], [py, py],
                        color="#94a3b8", lw=1, ls=(0, (2, 2)), zorder=2)
            for arm, color in (("rbd5", "#2d5fa6"), ("rbd3", "#c05820")):
                for segment_start, segment_end in arm_segments[arm]:
                    ax.plot([segment_start, segment_end], [py, py], color=color, lw=5,
                            solid_capstyle="butt", zorder=3)
                    ax.scatter((segment_start + segment_end) / 2, py, marker="s", s=26,
                               color=color, zorder=4)
            probe_edges = [float(row[column]) for column in coordinate_columns]
            midpoint = min(probe_edges) + (max(probe_edges) - min(probe_edges)) / 2
            if not math.isfinite(midpoint):
                continue
            ax.annotate(_probe_label(row, label_mode=label_mode), (midpoint, py), xytext=(0, -8),
                        textcoords="offset points", ha="center", va="top", fontsize=7.5,
                        bbox={"facecolor": "white", "edgecolor": "none", "alpha": .82, "pad": 1})
    # BED panels have exact block geometry but may not yet have a transcript
    # assignment. Put them on explicit panel tracks instead of dropping them.
    for offset, (_, row) in enumerate(bed_only.head(20).iterrows()):
        py = -.65 - .22 * offset
        starts = row.get("bed_block_starts")
        sizes = row.get("bed_block_sizes")
        if isinstance(starts, str):
            starts = [int(value) for value in starts.rstrip(",").split(",") if value]
        if isinstance(sizes, str):
            sizes = [int(value) for value in sizes.rstrip(",").split(",") if value]
        if not isinstance(starts, (list, tuple)) or not isinstance(sizes, (list, tuple)):
            starts, sizes = [0], [int(row["chrom_end"] - row["chrom_start"])]
        for block_start, block_size in zip(starts, sizes):
            try:
                block_start, block_size = float(block_start), float(block_size)
            except (TypeError, ValueError):
                continue
            if not math.isfinite(block_start) or not math.isfinite(block_size) or block_size <= 0:
                continue
            start = float(row["chrom_start"]) + block_start
            ax.plot([start, start + block_size], [py, py], color="#7c3aed", lw=5,
                    solid_capstyle="butt", zorder=3)
            ax.scatter(start + block_size / 2, py, marker="D", s=28,
                       color="#7c3aed", edgecolor="white", linewidth=.5, zorder=4)
        ax.annotate(_probe_label(row, label_mode=label_mode),
                    ((float(row["chrom_start"]) + float(row["chrom_end"])) / 2, py),
                    xytext=(0, -8), textcoords="offset points", ha="center", va="top",
                    fontsize=7.5, color="#5b21b6",
                    bbox={"facecolor": "white", "edgecolor": "none", "alpha": .82, "pad": 1})
    ax.set_yticks([])
    ax.set_xlim(view_left, view_right)
    ax.set_ylim(-.92 - .22 * len(bed_only.head(20)), len(models) - .3)
    from matplotlib.ticker import FuncFormatter
    coordinate_formatter = FuncFormatter(lambda value, _: f"{int(value):,}")
    ax.xaxis.set_major_formatter(coordinate_formatter)
    ax.set_xlabel("genomic coordinate (0-based; bp)")
    if overview_enabled:
        fig.suptitle(title or f"{gene}: isoforms and mapped probes",
                     fontweight="bold", x=.01, ha="left")
        count_label = f"{len(models)} probe-supporting transcript" + ("" if len(models) == 1 else "s")
        ax.set_title(f"Zoomed genomic view — the same {count_label}",
                     fontsize=9, loc="left", color="#475569", pad=8)
    else:
        ax.set_title(title or f"{gene}: isoforms and mapped probes",
                     fontweight="bold", loc="left", pad=16)
    ax.spines[["top", "right", "left"]].set_visible(False)
    handles = [plt.Line2D([0], [0], color="#475569", lw=5, label="exon")]
    if overview_enabled and not mapped.empty:
        handles.append(plt.Line2D([0], [0], color="#0f766e", marker="o", lw=1.5,
                                  label="custom/designed junction arc"))
    if not bed_only.empty:
        handles.append(plt.Line2D([0], [0], color="#7c3aed", marker="D", lw=3,
                                  label="existing 10x probe"))
    if not mapped.empty:
        handles.extend([
            plt.Line2D([0], [0], color="#2d5fa6", marker="s", lw=3, label="RBD5"),
            plt.Line2D([0], [0], color="#c05820", marker="s", lw=3, label="RBD3"),
        ])
    legend_ax.legend(handles=handles, frameon=False, ncol=min(5, len(handles)), loc="center")
    if overview_ax is not None:
        locus_left = min(exon[0] for _, _, exons, _ in models for exon in exons)
        locus_right = max(exon[1] for _, _, exons, _ in models for exon in exons)
        all_model_exons = [exon for _, _, exons, _ in models for exon in exons]
        transform, _, _, intron_scale = _compressed_locus_transform(all_model_exons)
        use_compressed = overview_scale == "compressed" or (
            overview_scale == "auto" and intron_scale < .05
        )
        display = transform if use_compressed else lambda value: float(value)
        display_left, display_right = display(locus_left), display(locus_right)
        overview_rows = {}
        overview_exons = {}
        for y, (transcript_id, label, exons, strand) in enumerate(models[::-1]):
            overview_rows[str(transcript_id)] = y
            overview_exons[str(transcript_id)] = exons
            transcript_left, transcript_right = display(exons[0][0]), display(exons[-1][1])
            overview_ax.plot([transcript_left, transcript_right], [y, y],
                             color="#cbd5e1", lw=1, zorder=0)
            for start, end in exons:
                shown_start, shown_end = display(start), display(end)
                overview_ax.add_patch(plt.Rectangle((shown_start, y - .12), shown_end - shown_start, .24,
                                                    color="#475569", ec="none", zorder=1))
            overview_ax.text(display_left + .006 * max(display_right - display_left, 1), y + .17,
                             _transcript_label(label, strand), fontsize=7,
                             ha="left", va="bottom")
        shown_view_left = display(max(view_left, locus_left))
        shown_view_right = display(min(view_right, locus_right))
        overview_ax.axvspan(shown_view_left, shown_view_right, color="#f59e0b", alpha=.14, zorder=0)
        overview_ax.axvline(shown_view_left, color="#d97706", lw=.8, ls="--")
        overview_ax.axvline(shown_view_right, color="#d97706", lw=.8, ls="--")
        if not mapped.empty:
            for _, row in mapped.iterrows():
                transcript_y = overview_rows.get(str(row.get("transcript_id")))
                if transcript_y is None:
                    continue
                arm_left = min(row["rbd5_chrom_start"], row["rbd3_chrom_start"])
                arm_right = max(row["rbd5_chrom_end"], row["rbd3_chrom_end"])
                touched = [
                    (max(arm_left, start), min(arm_right, end))
                    for start, end in overview_exons[str(row.get("transcript_id"))]
                    if max(arm_left, start) < min(arm_right, end)
                ]
                positions = [display((start + end) / 2) for start, end in touched]
                if len(positions) >= 2:
                    from matplotlib.path import Path as MplPath
                    from matplotlib.patches import PathPatch
                    left, right = min(positions), max(positions)
                    apex_y = transcript_y - .34
                    path = MplPath(
                        [(left, transcript_y - .08), ((left + right) / 2, apex_y),
                         (right, transcript_y - .08)],
                        [MplPath.MOVETO, MplPath.CURVE3, MplPath.CURVE3],
                    )
                    overview_ax.add_patch(PathPatch(
                        path, facecolor="none", edgecolor="#0f766e", lw=1.6, zorder=3,
                    ))
                    overview_ax.scatter((left + right) / 2, apex_y, marker="o", s=18,
                                        color="#0f766e", edgecolor="white", linewidth=.4, zorder=4)
                elif positions:
                    overview_ax.scatter(positions[0], transcript_y - .2, marker="o", s=28,
                                        color="#0f766e", edgecolor="white", linewidth=.5, zorder=3)
        if not bed_only.empty:
            existing_midpoints = (bed_only["chrom_start"] + bed_only["chrom_end"]) / 2
            overview_ax.scatter([display(value) for value in existing_midpoints],
                                [-.55] * len(existing_midpoints),
                                marker="D", s=24, color="#7c3aed", label="existing 10x", zorder=3)
        overview_ax.set_xlim(display_left, display_right)
        overview_ax.set_ylim(-.72, len(models) - .3)
        overview_ax.set_yticks([])
        if use_compressed:
            overview_ax.set_xticks([])
            overview_ax.set_xlabel("")
        else:
            overview_ax.xaxis.set_major_formatter(coordinate_formatter)
        overview_ax.tick_params(axis="x", labelsize=7, colors="#64748b")
        scale_note = "introns compressed" if use_compressed else "genomic scale"
        overview_ax.set_title(f"Full-length view — {len(models)} probe-supporting transcript"
                              f"{'s' if len(models) != 1 else ''} ({scale_note}); shaded region is detailed below",
                              fontsize=9, loc="left", color="#475569")
        overview_ax.spines[["top", "right", "left"]].set_visible(False)
    _save_figure(fig, save, dpi=dpi, transparent=transparent)
    return fig

In [ ]:
#| export
def design_comparison(data, reference_score: str, designed_score: str, *,
                      label: str = "probe_name", same_sequence: str | None = None,
                      reference_label: str = "external control",
                      designed_label: str = "selected design",
                      title: str | None = None, figsize=None,
                      save: str | Path | None = None, dpi: int = 300,
                      transparent: bool = False):
    """Paired comparison of reference and independently selected probe scores."""
    plt = _plt()
    frame = _frame(data).copy()
    required = {label, reference_score, designed_score}
    if same_sequence:
        required.add(same_sequence)
    missing = required - set(frame)
    if missing:
        raise ValueError(f"Missing comparison columns: {sorted(missing)}")
    frame = frame.dropna(subset=[reference_score, designed_score]).copy()
    if frame.empty:
        return _empty_figure("No comparable probe scores", figsize or (8, 2.5))
    frame["_difference"] = frame[designed_score] - frame[reference_score]
    frame = frame.sort_values(["_difference", label], ascending=[False, True]).reset_index(drop=True)
    y = list(range(len(frame)))
    fig, ax = plt.subplots(
        figsize=figsize or (9, max(3.2, .52 * len(frame) + 1.5)),
        layout="constrained",
    )
    for position, (_, row) in zip(y, frame.iterrows()):
        ax.plot([row[reference_score], row[designed_score]],
                [position - .08, position + .08], color="#cbd5e1", lw=1.5, zorder=1)
    ax.scatter(frame[reference_score], [value - .08 for value in y], marker="o", s=42,
               color="#64748b", edgecolor="white", linewidth=.5,
               label=reference_label, zorder=2)
    ax.scatter(frame[designed_score], [value + .08 for value in y], marker="D", s=44,
               color="#0f766e", edgecolor="white", linewidth=.5,
               label=designed_label, zorder=3)
    ax.set_yticks(y, frame[label].astype(str))
    ax.invert_yaxis()
    values = pd.concat([frame[reference_score], frame[designed_score]])
    span = max(float(values.max() - values.min()), .1)
    left = max(0.0, float(values.min()) - .12 * span)
    right = min(1.0, float(values.max()) + .28 * span)
    if right <= float(values.max()):
        right = float(values.max()) + .08 * span
    ax.set_xlim(left, right)
    annotation_x = right - .015 * (right - left)
    for position, (_, row) in zip(y, frame.iterrows()):
        difference = float(row["_difference"])
        sequence_note = ""
        if same_sequence:
            sequence_note = " · same RBD" if bool(row[same_sequence]) else " · alternative RBD"
        ax.text(annotation_x, position, f"Δ {difference:+.4f}{sequence_note}",
                ha="right", va="center", fontsize=7,
                color="#0f766e" if difference > 0 else "#64748b")
    ax.set_xlabel("prospective ProbeHarbour score")
    ax.set_title(title or "External control versus selected design", loc="left", fontweight="bold")
    ax.grid(axis="x", color="#e2e8f0", linewidth=.7)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.legend(frameon=False, ncol=1, loc="center left", bbox_to_anchor=(1.01, .5))
    _save_figure(fig, save, dpi=dpi, transparent=transparent)
    return fig

In [ ]:
#| export
def candidate_card(data, candidate=None, *, title: str | None = None,
                   subtitle: str | None = None, figsize=(10, 7.4),
                   save: str | Path | None = None, dpi: int = 300,
                   transparent: bool = False):
    """Render one candidate as a self-contained, auditable record card.

    ``candidate`` may be a dataframe index, candidate ID, probe name, or row.
    Missing analyses are labelled ``not calculated`` rather than inferred.
    """
    plt = _plt()
    from matplotlib.patches import FancyBboxPatch
    frame = _frame(data)
    if isinstance(candidate, pd.Series):
        row = candidate
    elif candidate is None:
        if frame.empty:
            return _empty_figure("No candidate records", figsize)
        if len(frame) != 1:
            raise ValueError("candidate is required when data contains more than one row")
        row = frame.iloc[0]
    elif candidate in frame.index:
        row = frame.loc[candidate]
    else:
        matches = pd.Series(False, index=frame.index)
        for column in ("candidate_id", "probe_name"):
            if column in frame:
                matches |= frame[column].astype(str).eq(str(candidate))
        if matches.sum() != 1:
            raise KeyError(f"Expected one candidate matching {candidate!r}; found {int(matches.sum())}")
        row = frame.loc[matches].iloc[0]

    rbd5, rbd3 = str(_value(row, "rbd5", default="")), str(_value(row, "rbd3", default=""))
    stored_ligation = _value(row, "lig_dinuc", "ligation_dinucleotide")
    ligation, ligation_warning = _ligation_join(rbd5, rbd3, stored_ligation)
    name = _value(row, "probe_name", "candidate_id", default="candidate")
    gene = _value(row, "gene_name", "gene", "gene_id", default="unknown gene")
    transcript = _value(row, "transcript_id", default="not assigned")
    junction = _value(row, "junction_id", default="not assigned")
    role = _value(row, "panel_role", "source", default="candidate")
    score = _value(row, "probeharbour_score", "recalculated_score", "raw_design_score",
                   "stored_raw_design_score", "score")
    base_score = _value(row, "probeharbour_score_base_score", "recalculated_score_base_score",
                        "base_score")
    specificity_score = _value(row, "specificity_score")
    specificity_percent = _value(row, "specificity_percent_recalculated")
    if specificity_score is not None:
        specificity_label = "specificity score (0–1)"
        specificity_display = _present(specificity_score, digits=4)
    else:
        specificity_label = "specificity score"
        specificity_display = _present(specificity_percent, digits=1, suffix="%")
    specificity_evidence = _value(row, "specificity_evidence", default={})
    if isinstance(specificity_evidence, str):
        try:
            specificity_evidence = json.loads(specificity_evidence)
        except json.JSONDecodeError:
            specificity_evidence = {}
    if not isinstance(specificity_evidence, dict):
        specificity_evidence = {}
    score_modifiers = _value(row, "probeharbour_score_modifiers", "score_modifiers")
    readable_modifiers = _readable_items(score_modifiers, {
        "does_not_meet_tm_constraints": "full-binder Tm outside chemistry window",
        "has_simple_repeat": "simple repeat",
        "has_5g_repeat": "5G/C-run",
        "potential_homodimer": "strong predicted homodimer",
    })
    modifier_count = (0 if readable_modifiers == "none" else
                      (1 if isinstance(score_modifiers, str) else len(score_modifiers)))
    score_calculation = (f"{_present(base_score, digits=4)} / (1 + {modifier_count} flag"
                         f"{'s' if modifier_count != 1 else ''})")
    compatibility_reasons = _readable_items(
        _value(row, "panel_compatibility_reasons"), {
            "duplicate_binder": "duplicate binder",
            "reverse_complement_binder": "reverse-complement binder",
            "duplicate_arm": "duplicate arm",
            "long_complementary_run": "long complementary run",
            "strong_panel_heterodimer": "strong panel heterodimer",
            "existing_panel_spacing": "<75 bp from existing probe",
            "panel_spacing_violation": "<75 bp from another add-on",
        },
    )
    existing_compatibility_reasons = _readable_items(
        _value(row, "existing_panel_compatibility_reasons"), {
            "duplicate_binder": "duplicate binder", "duplicate_arm": "duplicate arm",
            "reverse_complement_binder": "reverse-complement binder",
            "long_complementary_run": "long complementary run",
            "strong_panel_heterodimer": "strong heterodimer",
            "existing_panel_spacing": "spacing <75 bp",
        },
    )
    coordinates = _value(row, "coordinates", default={})
    coordinates = coordinates if isinstance(coordinates, dict) else {}
    chrom = _value(row, "chrom", "chromosome", default=coordinates.get("chrom"))
    start = _value(row, "chrom_start", default=coordinates.get("start"))
    end = _value(row, "chrom_end", default=coordinates.get("end"))
    strand = _value(row, "strand", default=coordinates.get("strand", "?"))
    r5_start = _value(row, "rbd5_chrom_start", default=coordinates.get("rbd5_start"))
    r5_end = _value(row, "rbd5_chrom_end", default=coordinates.get("rbd5_end"))
    r3_start = _value(row, "rbd3_chrom_start", default=coordinates.get("rbd3_start"))
    r3_end = _value(row, "rbd3_chrom_end", default=coordinates.get("rbd3_end"))
    interval = "not mapped" if chrom is None or start is None or end is None else f"{chrom}:{int(start):,}–{int(end):,} ({strand})"
    if all(value is not None for value in (chrom, r5_start, r5_end, r3_start, r3_end)):
        r5_block = f"{chrom}:{int(r5_start):,}–{int(r5_end):,} ({strand})"
        r3_block = f"{chrom}:{int(r3_start):,}–{int(r3_end):,} ({strand})"
    else:
        r5_block = r3_block = "not mapped"
    junction_spanning = _value(row, "junction_spanning")
    junction_relationship = _value(row, "junction_relationship")
    if junction_relationship is None and junction_spanning is not None:
        junction_relationship = "junction-spanning" if bool(junction_spanning) else "within exon"
    database_id = specificity_evidence.get("database_id")
    database_version = specificity_evidence.get("database_version")
    if database_id is None:
        database_display = "not calculated"
    elif database_version is None:
        database_display = _present(database_id)
    else:
        database_display = f"{_present(database_id)} · v{_present(database_version)}"

    def neighbour(value):
        value = _present(value)
        match = re.search(r":J(\d+):slide:([+-]\d+)$", value)
        return f"J{match.group(1)}, shift {match.group(2)} nt" if match else _short_identifier(value, 32)

    coverage = _present(_value(row, "target_coverage"), digits=1, suffix="%")
    supporting = _value(row, "junction_transcript_count")
    annotated = _value(row, "gene_transcript_count")
    coverage_display = (f"{coverage} · {int(supporting)}/{int(annotated)} transcripts"
                        if supporting is not None and annotated is not None else coverage)

    fig = plt.figure(figsize=figsize, layout="constrained")
    ax = fig.add_subplot(111); ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ink, muted, edge, face = "#21313a", "#61727b", "#c8d3d9", "#f8fafb"

    def box(x, y, width, height, heading, rows):
        ax.add_patch(FancyBboxPatch((x, y), width, height,
                     boxstyle="round,pad=.008,rounding_size=.012",
                     facecolor=face, edgecolor=edge, lw=.8))
        ax.text(x + .018, y + height - .035, heading, va="top", fontweight="bold",
                fontsize=10, color=ink)
        step = (height - .085) / max(len(rows), 1)
        for index, (label, value) in enumerate(rows):
            yy = y + height - .085 - index * step
            ax.text(x + .018, yy, label, va="top", fontsize=7, color=muted)
            ax.text(x + width * .43, yy, str(value), va="top", fontsize=7.15,
                    color=ink, fontweight="medium", wrap=True)

    ax.text(.055, .955, title or "ProbeStore candidate record", fontsize=15,
            fontweight="bold", va="top", color="#18262e")
    ax.text(.055, .918, subtitle or "Sequence, target context, scoring and available validation evidence",
            fontsize=8, va="top", color=muted)
    ax.add_patch(FancyBboxPatch((.055, .805), .89, .075,
                 boxstyle="round,pad=.008,rounding_size=.012",
                 facecolor="#eef3f5", edgecolor=edge, lw=.8))
    ax.text(.07, .855, f"{gene}  •  {name}", fontsize=11, fontweight="bold", va="center", color=ink)
    ax.text(.07, .825, f"Transcript {transcript}  |  junction {junction}  |  role: {role}",
            fontsize=7.5, va="center", color="#52616a")
    ax.text(.055, .772, "Binding architecture", fontsize=10, fontweight="bold", color=ink)
    boundary = _draw_binding_arms(
        ax, x=.055, y=.707, width=.89, height=.045, rbd5=rbd5, rbd3=rbd3,
    )
    ax.text(boundary, .775, f"ligation boundary  {_present(ligation)}",
            ha="center", va="bottom", fontsize=7.3, color="#0f766e", fontweight="bold")
    shift = _value(row, "shift_nt", "junction_shift", "shift")
    join_note = f"terminal bases {_present(ligation)} (RBD5 last | RBD3 first)"
    if ligation_warning:
        join_note += f"  |  WARNING: {ligation_warning}"
    ax.text(.055, .648, f"arm lengths {len(rbd5)} / {len(rbd3)} nt  |  shift {_present(shift, suffix=' nt')}  |  {join_note}",
            fontsize=8, color=ink)

    box(.055, .405, .43, .225, "Target and genomic context", [
        ("gene / transcript", f"{gene} / {transcript}"), ("junction identity", junction),
        ("transcript ligation site", f"{_present(_value(row, 'mapped_cut', 'ligation_site'))} (0-based nt)"),
        ("genomic 5′ RBD block", r5_block),
        ("genomic 3′ RBD block", r3_block),
        ("junction relationship", _present(junction_relationship)),
        ("junction class", _present(_value(row, "junction_specificity_class", "junction_class"))),
    ])
    box(.515, .405, .43, .225, "Thermodynamics and intrinsic QC", [
        ("Tm: 5′ / 3′ / full", " / ".join(_present(_value(row, key), digits=1, suffix=" °C") for key in ("tm_5p", "tm_3p", "tm_full"))),
        ("GC content", _present(_value(row, "gc_percent", "gc_pct"), digits=1, suffix="%")),
        ("Tm / repeat / 5G", f"{_modifier_status(row, 'does_not_meet_tm_constraints', pass_field='pass_tm_full')} / {_modifier_status(row, 'has_simple_repeat', pass_field='pass_repeat', fail_field='has_simple_repeat')} / {_modifier_status(row, 'has_5g_repeat', fail_field='has_5g_repeat')}"),
        ("homodimer", _modifier_status(row, "potential_homodimer", pass_field="pass_p3_homodimer")),
        ("panel compatibility", _status(_value(row, "panel_compatibility_pass", "pass_panel_spacing"))),
    ])
    box(.055, .105, .43, .255, "Design score and selection inputs", [
        ("ProbeHarbour score (0–1)", _present(score, digits=4)),
        ("score calculation", score_calculation),
        ("score modifiers", readable_modifiers),
        ("annotation coverage", coverage_display),
        ("existing 10x neighbour", f"{_status(_value(row, 'existing_panel_compatibility_pass'))} · {_present(_value(row, 'nearest_existing_panel_probe_distance'), digits=0, suffix=' bp')} · {neighbour(_value(row, 'nearest_existing_panel_probe_name'))}"),
        ("existing 10x findings", existing_compatibility_reasons),
        ("add-on neighbour", f"{_status(_value(row, 'pass_panel_spacing'))} · {_present(_value(row, 'nearest_probe_distance'), digits=0, suffix=' bp')} · {neighbour(_value(row, 'nearest_probe_name'))}"),
        ("add-on findings", compatibility_reasons),
    ])
    box(.515, .105, .43, .255, "Specificity and off-target evidence", [
        (specificity_label, specificity_display),
        ("target / best-off bitscore", f"{_present(_value(row, 'raw_blast_score', default=specificity_evidence.get('target_raw_score')), digits=1)} / {_present(_value(row, 'best_offtarget_raw_score', default=specificity_evidence.get('best_offtarget_raw_score')), digits=1)}"),
        ("off-target alignments", _present(_value(row, 'off_targets', 'off_target_count', default=specificity_evidence.get('off_target_count')))),
        ("best off-target", _short_identifier(_value(row, 'best_off_target', default=specificity_evidence.get('best_offtarget_id')))),
        ("reference database", database_display),
        ("method", _present(specificity_evidence.get("method_id"))),
    ])
    ax.text(.055, .045, "A card reports what the ProbeStore contains for this candidate. Missing downstream analyses are shown explicitly and can be added without regenerating the candidate pool.",
            fontsize=7.5, color="#52616a", wrap=True)
    _save_figure(fig, save, dpi=dpi, transparent=transparent)
    return fig

In [ ]:
#| export
def candidate_cards(data, *, title: str = "ProbeStore candidate comparison",
                    card_title: str = "probe_name", accent: str = "#0f766e",
                    max_cards: int = 6, ncols: int = 2, figsize=None,
                    save: str | Path | None = None, dpi: int = 300,
                    transparent: bool = False):
    """Render a compact grid for comparing several candidate records."""
    plt = _plt()
    from matplotlib.patches import FancyBboxPatch
    frame = _frame(data).head(max_cards)
    if frame.empty:
        return _empty_figure("No candidate records", figsize or (9, 3))
    ncols = max(1, min(int(ncols), len(frame))); nrows = math.ceil(len(frame) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize or (5 * ncols, 3.4 * nrows), squeeze=False)
    for ax, (_, row) in zip(axes.flat, frame.iterrows()):
        ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.add_patch(FancyBboxPatch((.01, .01), .98, .98, boxstyle="round,pad=.008,rounding_size=.02",
                                   facecolor="#f8fafb", edgecolor="#c8d3d9", lw=.8))
        ax.add_patch(FancyBboxPatch((.01, .86), .98, .13, boxstyle="round,pad=.008,rounding_size=.02",
                                   facecolor=accent, edgecolor=accent, lw=.8))
        heading = _value(row, card_title, "candidate_id", default="candidate")
        ax.text(.045, .925, str(heading), color="white", fontsize=10, fontweight="bold", va="center")
        ax.text(.045, .81, f"{_value(row, 'gene_name', 'gene', 'gene_id', default='unknown gene')}  |  {_value(row, 'junction_id', default='junction not assigned')}", fontsize=7.3, color="#34444d")
        r5, r3 = str(_value(row, "rbd5", default="")), str(_value(row, "rbd3", default=""))
        boundary = _draw_binding_arms(
            ax, x=.045, y=.66, width=.91, height=.08, rbd5=r5, rbd3=r3,
            show_labels=False, sequence_font=7,
        )
        stored_ligation = _value(row, "lig_dinuc", "ligation_dinucleotide")
        ligation, ligation_warning = _ligation_join(r5, r3, stored_ligation)
        ax.text(boundary, .755, f"ligation boundary  {_present(ligation)}",
                ha="center", va="bottom", fontsize=6.6, color="#0f766e", fontweight="bold")
        values = [
            ("score (0–1)", _present(_value(row, "probeharbour_score", "recalculated_score", "raw_design_score", "stored_raw_design_score"), digits=4)),
            ("shift / terminal bases", f"{_present(_value(row, 'shift_nt', 'junction_shift', 'shift'), suffix=' nt')} / {_present(ligation)}" + (f" ⚠ {ligation_warning}" if ligation_warning else "")),
            ("full-binder Tm", _present(_value(row, "tm_full"), digits=1, suffix=" °C")),
            ("specificity", _present(_value(row, "specificity_score", "specificity_percent_recalculated"), digits=1, suffix="%")),
            ("sequence QC", f"repeat {_modifier_status(row, 'has_simple_repeat', pass_field='pass_repeat', fail_field='has_simple_repeat')}; 5G {_modifier_status(row, 'has_5g_repeat', fail_field='has_5g_repeat')}"),
            ("panel compatibility", _status(_value(row, "panel_compatibility_pass", "pass_panel_spacing"))),
        ]
        for index, (label, value) in enumerate(values):
            yy = .57 - index * .082
            ax.text(.05, yy, label, fontsize=7, color="#61727b", va="center")
            ax.text(.34, yy, value, fontsize=7.3, color="#21313a", fontweight="medium", va="center")
    for ax in axes.flat[len(frame):]: ax.set_axis_off()
    fig.suptitle(title, x=.02, ha="left", fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, .95))
    _save_figure(fig, save, dpi=dpi, transparent=transparent)
    return fig

In [ ]:
#| export
def score_comparison(data, score_x: str, score_y: str, *, label_x=None, label_y=None,
                     color_by: str | None = None, title: str | None = None, figsize=(5, 4.5)):
    """Plot two scores with identity line, Pearson correlation, and sample size."""
    plt = _plt()
    frame = _frame(data)
    needed = [score_x, score_y] + ([color_by] if color_by else [])
    missing = set(needed) - set(frame)
    if missing:
        raise ValueError(f"Missing plotting columns: {sorted(missing)}")
    values = frame[needed].dropna(subset=[score_x, score_y])
    if values.empty:
        return _empty_figure("No overlapping scores", figsize)
    fig, ax = plt.subplots(figsize=figsize)
    if color_by:
        for category, group in values.groupby(color_by, dropna=False):
            ax.scatter(group[score_x], group[score_y], s=18, alpha=.6, label=str(category))
        ax.legend(frameon=False, fontsize=7)
    else:
        ax.scatter(values[score_x], values[score_y], s=18, alpha=.55, color="#2563eb")
    lo = min(values[score_x].min(), values[score_y].min())
    hi = max(values[score_x].max(), values[score_y].max())
    ax.plot([lo, hi], [lo, hi], "--", color="#94a3b8", linewidth=1)
    correlation = values[[score_x, score_y]].corr().iloc[0, 1]
    ax.text(.03, .97, f"n={len(values):,}\nPearson r={correlation:.3f}", transform=ax.transAxes, ha="left", va="top")
    ax.set(xlabel=label_x or score_x, ylabel=label_y or score_y,
           title=title or f"{label_y or score_y} vs {label_x or score_x}")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    return fig